In [ ]:
// Nested tuples, the representation of abstract syntax trees; loaded once per session.
#load "../Support/Nested.fsx"

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"

In [ ]:
let divide (numerator: float) (denominator: float) =
    if denominator = 0.0 then failwith "Division by zero"
    numerator / denominator

# A Simple Calculator Using EBNF

This file shows the implementation of a *symbolic calculator* using `Larkita`.  It is a variant of the notebook
`01-Calculator.ipynb`:  Instead of left recursion, the grammar uses the repetition operator `*` of the
*extended Backus-Naur form* (EBNF).  The grammar for the language implemented by this parser is as follows:
$$
\begin{array}{lcl}
  \texttt{stmnt}   & \rightarrow & \;\texttt{IDENTIFIER}\; \texttt{':='} \; \;\texttt{expr}\; \texttt{';'} \\
                   & \mid        & \;\texttt{expr}\; \texttt{';'} \\[0.2cm]
  \texttt{expr}    & \rightarrow & \;\texttt{product}\; \bigl((\texttt{'+'} \mid \texttt{'-'})\; \texttt{product}\bigr)^* \\[0.2cm]
  \texttt{product} & \rightarrow & \;\texttt{factor}\; \bigl((\texttt{'*'} \mid \texttt{'/'})\; \texttt{factor}\bigr)^* \\[0.2cm]
  \texttt{factor}  & \rightarrow &   \texttt{'('} \; \texttt{expr} \;\texttt{')'}      \\
                   & \mid        & \;\texttt{NUMBER}                                   \\
                   & \mid        & \;\texttt{IDENTIFIER}
\end{array}
$$
The abstract syntax tree and its evaluation are the same as in `01-Calculator.ipynb`.

## Setup and Imports
We load the F# port of `Larkita`.  It generates a *canonical LR(1) parser* that returns the abstract syntax tree
of its input as a *nested tuple*, i.e. as a value of the type `nested` (see `01-Calculator.ipynb`).

In [ ]:
#load "../Support/Larkita/Larkita.fsx"
open Larkita
open Nested

## Specification of the Grammar

`Larkita` reads grammars in the *extended Backus-Naur Form* (EBNF) of `Lark`:  both the lexical tokens and the
grammar rules are defined in a single grammar.  It is stored in the file `calc-ebnf.lark`, which the cell below
displays.  Let us break down the components of our grammar definition:

* `stmnt`: A statement is either an assignment or an expression followed by a semicolon.  The first rule of the
   grammar defines the start symbol.  As explained in `01-Calculator.ipynb`, the string literals of an
   alternative form the *head* of the tuple that is returned, so the two alternatives yield
   `((":=", ";"), ("IDENTIFIER", name), ast(expr))` and `(";", ast(expr))`.
* **Repetition** `( ... )*`: The rule `expr: product (("+" | "-") product)*` states that an expression is a product
   that is followed by zero or more occurrences of an additive operator and another product.  This single rule replaces
   the three left-recursive rules
   ```
   expr: expr "+" product
       | expr "-" product
       | product
   ```
   of `01-Calculator.ipynb`.  The rule for `product` is built the same way.  The hierarchy `expr`, `product`,
   `factor` still enforces operator precedence (e.g., multiplication in `product` happens before addition in `expr`).
* **Flat tuples**: The head of a tuple is determined by the grammar alone, not by the input.  The operands and
   the operators are collected in a single *flat* tuple, for example `1 - 2 + 3` yields
   `("expr", p1, "-", p2, "+", p3)`, where `p1`, `p2`, and `p3` are the tuples representing the products.  Since the head is fixed by the grammar, a single product yields the tuple
   `("expr", p1)`, too.  Hence the grammar no longer specifies that `-` associates to the left.  We take care of
   this when we build the AST.
* **Inner Alternatives** `("+" | "-")`: Outside of a repetition, an inner alternative is an abbreviation, e.g.
   `expr: expr ("+" | "-") product` stands for the two alternatives `expr "+" product` and `expr "-" product`, so
   the operator that has been read is the head.  Inside a repetition this is not possible, since a single tuple
   collects all operands and its head is fixed by the grammar.  Therefore, the alternative
   `product (("+" | "-") product)*` contains no literal that belongs to the head and the name of the rule is used
   as head.  The operator that has been read becomes a child of the tuple instead.  As it is a literal, it is
   represented by a plain string like `"-"`, i.e. by the leaf `Leaf (Str "-")`, not by a pair.
* `%ignore "(", ")"` removes the parentheses from the head of the alternative `"(" expr ")"`.  As this alternative
   then consists of a single symbol, it returns the abstract syntax tree of `expr`.
* **Lexical Imports** (`%import common...`): Instead of manually writing regular expressions for common tokens,
   we import standard terminal definitions.  Several imports are separated by commas.
   * `CNAME` parses standard variable names (letters, numbers, underscores) and is aliased to `IDENTIFIER`.
   * `NUMBER` handles both integers and floating-point numbers.
   * `WS` handles whitespace.
* `%ignore WS`: This directs the lexer to completely discard spaces and tabs so they do not clutter our parsing logic.

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "calc-ebnf.lark")

We generate the parser.  `Larkita` always generates a canonical LR(1) parser and writes a description of all
LR(1) states into the file specified by `outFile`.

In [ ]:
let parser = Larkita.Open("calc-ebnf.lark", outFile = "calc-ebnf.out")

The nested tuple returned for `1 - 2 * 3 - 4;` shows the flat tuples:  after the head `"expr"`, operands and
operators alternate.

In [ ]:
printfn "%s" (toString (parse parser "1 - 2 * 3 - 4;"))

## Generating the Abstract Syntax Tree (ADT)

Our evaluation function expects the *binary* tuples of `01-Calculator.ipynb`, where every operator has
exactly two operands.  For example, the string `x := y + z;` should be represented by the ADT
`((":=", ";"), ("IDENTIFIER", "x"), ("+", ("IDENTIFIER", "y"), ("IDENTIFIER", "z")))`.

A flat tuple with head `"expr"` or `"product"` has the form `(head, operand, op, operand, op, ..., operand)`.  The
function `to_binary` combines the operands from left to right, so `1 - 2 - 3` becomes
`("-", ("-", ("NUMBER", "1"), ("NUMBER", "2")), ("NUMBER", "3"))`.  The operators are strings like `"-"`, which are
exactly the heads of the binary tuples.  The tuples representing tokens are returned unchanged, while all other
tuples are converted recursively.  Therefore, the ADT is the same as in `01-Calculator.ipynb`.

In [ ]:
let rec toBinary (ast: nested) : nested =
    match ast with
    | Node [ Leaf (Str ("NUMBER" | "IDENTIFIER")); _ ] -> ast
    | Node (Leaf (Str ("expr" | "product")) :: first :: rest) ->
        // Combine the operands of a flat tuple from left to right
        let rec combine result rest =
            match rest with
            | Leaf (Str op) :: operand :: rest -> combine (Node [ Leaf (Str op); result; toBinary operand ]) rest
            | [] -> result
            | _ -> failwithf "toBinary: unexpected tuple %s" (toString ast)
        combine (toBinary first) rest
    | Node (head :: children) -> Node (head :: List.map toBinary children)
    | Leaf _ -> ast

In [ ]:
let binary = toBinary (parse parser "1 - 2 * 3 - 4;")
printfn "%s" (toString binary)
checkNotebook (toString binary = """(";", ("-", ("-", ("NUMBER", "1"), ("*", ("NUMBER", "2"), ("NUMBER", "3"))), ("NUMBER", "4")))""")

## Evaluating the Abstract Syntax Tree

Now that we have binary trees, we can use the same evaluation functions as in `01-Calculator.ipynb`.  The immutable
`names2Values` map associates variable names with values.

In [ ]:
let names2Values : Map<string, float> = Map.empty

In [ ]:
let rec evaluate names (adt: nested) : float =
    match adt with
    | Node [ Leaf (Str "NUMBER"); Leaf (Str value) ] -> System.Double.Parse(value, System.Globalization.CultureInfo.InvariantCulture)
    | Node [ Leaf (Str "IDENTIFIER"); Leaf (Str name) ] ->
        match Map.tryFind name names with
        | Some value -> value
        | None -> failwithf "Undefined variable: %s" name
    | Node [ Leaf (Str (("+" | "-" | "*" | "/") as op)); left; right ] ->
        let l, r = evaluate names left, evaluate names right
        match op with
        | "+" -> l + r
        | "-" -> l - r
        | "*" -> l * r
        | _ -> divide l r
    | _ -> failwithf "Unknown ADT structure: %s" (toString adt)

let execute names (adt: nested) =
    match adt with
    | Node [ Node [ Leaf (Str ":="); Leaf (Str ";") ]; Node [ Leaf (Str "IDENTIFIER"); Leaf (Str name) ]; expr ] -> Map.add name (evaluate names expr) names, None
    | Node [ Leaf (Str ";"); expr ] -> names, Some(evaluate names expr)
    | _ -> failwithf "Unknown ADT structure: %s" (toString adt)

let calculate names source = execute names (toBinary (parse parser source))

## Execution

The REPL (Read-Eval-Print Loop) takes the input string, parses it into an abstract syntax tree, converts the flat tuples into
binary tuples, and then evaluates the result.

In [ ]:
// Kept separate from the demonstrations so running all cells never blocks for input.
let main () =
    let rec loop names =
        printf "calc> "
        let source = System.Console.ReadLine()
        if not (System.String.IsNullOrEmpty source) then
            try
                let updated, value = calculate names source
                value |> Option.iter (printfn "%g")
                loop updated
            with
            | UnexpectedInput e -> printfn "Error: %s" e.Message; loop names
            | error -> printfn "Error: %s" error.Message; loop names
    loop Map.empty

In [ ]:
let assigned, _ = calculate names2Values "x := 1 - 2 - 3;"
let names, result = calculate assigned "x * 2 / 4;"
checkNotebook (result = Some -2.0)
printfn "%A" result

Check the populated memory map:

In [ ]:
printfn "%A" names

We can even get the parser states:  `Larkita` has written them into the file `calc-ebnf.out`.  As LR parsing works
with plain BNF rules, the repetition `(("+" | "-") product)*` is translated into the auxiliary left-recursive rules
```
__expr_star_0: "+" product
             | "-" product
             | __expr_star_0 "+" product
             | __expr_star_0 "-" product
```
and `expr: product __expr_star_0 | product`, which show up in the states.  An auxiliary rule does not build a tuple
of its own.  Instead, it collects its children in a list, and these children are spliced into the tuple of `expr`.
This is why the tuple is flat.  In the list of rules at the beginning of the file, the notation `*__expr_star_0`
stands for these spliced children, and `["-", ast(product)]` shows that the operator is a child.

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "calc-ebnf.out")